In [1]:
# Compare feature families, missingness, matching ratios, control reuse, and outcome horizons.
# Fit on training data, select settings on validation, and reserve test data for final evaluation.
# Write robustness_results.csv and model_performance_comparison.png.

In [2]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
# Load event data, saved chronological splits, and available feature families.
from pathlib import Path
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
                             brier_score_loss, confusion_matrix, f1_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/modeling_dataset_unprocessed.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
REPORTS = project_paths(ROOT).reports
FIGURES = project_paths(ROOT).figures
CATEGORICAL = ["sector", "industry"]
SPLITS = ["train", "validation", "test"]
MODEL_SETTINGS = {"C": 0.01, "l1_ratio": 0.9, "class_weight": "balanced",
                  "solver": "saga", "max_iter": 12000, "tol": 1e-4, "random_state": 42}

def read_events(path):
    frame = pd.read_csv(path, dtype={"company_id": "string"})
    frame["prediction_date"] = pd.to_datetime(frame["prediction_date"]).dt.normalize()
    frame["event_key"] = frame["company_id"].astype(str) + "|" + frame["prediction_date"].dt.strftime("%Y-%m-%d")
    return frame

events = read_events(PROCESSED / "modeling_dataset_unprocessed.csv")
if events["event_key"].duplicated().any():
    raise ValueError("Modeling data must have one row per company-event")
events = events.set_index("event_key", drop=False)
events["bankruptcy_date"] = pd.to_datetime(events["bankruptcy_date"], errors="coerce").dt.normalize()
events["metadata_outcome_end_date"] = pd.to_datetime(events["metadata_outcome_end_date"], errors="coerce").dt.normalize()
split_frames = [read_events(PROCESSED / f"{name}_ids.csv").assign(split=name) for name in SPLITS]
split_lookup = pd.concat(split_frames, ignore_index=True).set_index("event_key")["split"]
if split_lookup.index.duplicated().any():
    raise ValueError("Split files must assign each event exactly once")
events["split"] = split_lookup.reindex(events.index)

original = [c for c in events if c.startswith("original_")]
derived = [c for c in events if c.startswith("derived_")]
trends = [c for c in events if c.startswith("trend_")]
market = [c for c in events if c.startswith("market_")]
macro = [c for c in events if c.startswith("macro_")]
missing = [c for c in events if c.startswith("missing_")]
market_sources = {"adjusted_price", "market_cap", "shares_outstanding", "volume", "turnover"}
market_original = [c for c in original if c.removeprefix("original_") in market_sources
                   or c.startswith("original_market_monthly_price_")]
financial_original = [c for c in original if c not in market_original]
market_derived = [c for c in derived if c == "derived_market_to_book"]
financial_derived = [c for c in derived if c not in market_derived]
market_trends = [c for c in trends if any(c.startswith(f"trend_{name}_") for name in market_sources)
                 or "market_history" in c]
financial_trends = [c for c in trends if c not in market_trends]
financial = financial_original + financial_derived + financial_trends
market_features = market_original + market_derived + market_trends + market
all_values = original + derived + trends + market + macro
feature_sets = {
    "all_features": all_values,
    "financial_only": financial,
    "financial_plus_market": financial + market_features,
    "financial_plus_macro": financial + macro,
    "original_levels_only": original,
    "derived_ratios_only": derived,
    "trends_only": trends,
    "trends_1y": [c for c in trends if "_1y" in c or "has_1y" in c],
    "trends_2y": [c for c in trends if "_2y" in c or "has_2y" in c],
    "trends_3y": [c for c in trends if "_3y" in c or "has_3y" in c],
    "trends_1to3y": trends,
}
if any(not columns for columns in feature_sets.values()):
    raise ValueError("A requested feature family has no columns in the modeling dataset")
print(f"Loaded {len(events):,} events; feature families are ready.")


Loaded 768 events; feature families are ready.


C:\Users\MyPC\AppData\Local\Temp\ipykernel_13616\2018544025.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame["event_key"] = frame["company_id"].astype(str) + "|" + frame["prediction_date"].dt.strftime("%Y-%m-%d")
C:\Users\MyPC\AppData\Local\Temp\ipykernel_13616\2018544025.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  events["split"] = split_lookup.reindex(events.index)


In [3]:
# Build matched cohorts and the train-fitted modeling pipeline.
def event_key(company, dates):
    dates = pd.to_datetime(dates).dt.normalize()
    return company.astype(str) + "|" + dates.dt.strftime("%Y-%m-%d")

def matched_cohort(ratio, replacement):
    path = PROCESSED / f"matched_event_sample_1to{ratio}.csv"
    links = pd.read_csv(path, dtype={"bankrupt_company_id": "string", "control_company_id": "string"})
    links["bank_key"] = event_key(links["bankrupt_company_id"], links["bankrupt_prediction_date"])
    links["control_key"] = event_key(links["control_company_id"], links["control_pseudo_event_date"])
    links["bank_split"] = links["bank_key"].map(split_lookup)
    links["control_split"] = links["control_key"].map(split_lookup)
    links = links.loc[links["bank_split"].notna() & links["bank_split"].eq(links["control_split"])].copy()
    if not replacement:
        links = links.sort_values(["match_rank", "size_distance", "bankrupt_prediction_date"], kind="stable")
        links = links.drop_duplicates("control_company_id", keep="first")
    case_keys = links["bank_key"].drop_duplicates()
    control_counts = links["control_key"].value_counts()
    cases = events.reindex(case_keys).dropna(subset=["company_id"]).copy()
    controls = events.reindex(control_counts.index).dropna(subset=["company_id"]).copy()
    cases["sampling_weight"] = 1.0
    controls["sampling_weight"] = control_counts.reindex(controls.index).astype(float)
    cohort = pd.concat([cases, controls], axis=0).copy()
    cohort["matched_control_links"] = int(len(links))
    cohort["controls_per_case"] = len(links) / max(len(case_keys), 1)
    return cohort

def feature_columns(feature_set, include_missing=True):
    selected = list(dict.fromkeys(feature_sets[feature_set]))
    if include_missing:
        selected_bases = {c.removeprefix("original_") for c in selected if c.startswith("original_")}
        selected_bases.update(c.removeprefix("trend_").split("_change_")[0].split("_growth_")[0].split("_slope_")[0]
                              for c in selected if c.startswith("trend_"))
        selected.extend(c for c in missing if c.removeprefix("missing_") in selected_bases)
    return list(dict.fromkeys(selected))

def make_model(numeric_columns, C_value):
    categorical = [c for c in CATEGORICAL if c in events.columns]
    numeric_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
    ])
    category_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__MISSING__", keep_empty_features=True)),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ])
    prep = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", category_pipeline, categorical),
    ], remainder="drop")
    parameters = dict(MODEL_SETTINGS)
    parameters["C"] = C_value
    if inspect.signature(LogisticRegression).parameters["penalty"].default != "deprecated":
        parameters["penalty"] = "elasticnet"
    return Pipeline([("preprocess", prep), ("classifier", LogisticRegression(**parameters))])

def score(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    auc_roc = roc_auc_score(labels, probabilities) if np.unique(labels).size == 2 else np.nan
    return {
        "roc_auc": auc_roc,
        "pr_auc_average_precision": average_precision_score(labels, probabilities),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "f1": f1_score(labels, predictions, zero_division=0),
        "brier_score": brier_score_loss(labels, probabilities),
    }

# Each comparison changes one design choice while the other settings stay fixed.
scenarios = []
def add_scenario(name, comparison, feature_set="all_features", ratio=2, replacement=True,
                 horizon=12, include_missing=True):
    scenarios.append({"scenario": name, "comparison": comparison, "feature_set": feature_set,
                      "matching_ratio": ratio, "control_replacement": replacement,
                      "horizon_months": horizon, "missing_indicators": include_missing})

for name in ["financial_only", "financial_plus_market", "financial_plus_macro",
             "original_levels_only", "derived_ratios_only", "trends_only",
             "trends_1y", "trends_2y", "trends_3y", "trends_1to3y"]:
    add_scenario(name, "feature_family", feature_set=name)
add_scenario("all_features_with_missing_indicators", "missingness", include_missing=True)
add_scenario("all_features_without_missing_indicators", "missingness", include_missing=False)
for ratio in [1, 2, 3]:
    add_scenario(f"matching_1to{ratio}", "matching_ratio", ratio=ratio)
add_scenario("matching_1to2_without_replacement", "control_replacement", replacement=False)
for horizon in [6, 12]:
    add_scenario(f"outcome_{horizon}m", "outcome_horizon", horizon=horizon)
C_GRID = [0.01, 0.1, 1.0]
print(f"Configured {len(scenarios)} one-factor comparisons.")


Configured 18 one-factor comparisons.


In [4]:
# Select regularization on validation, evaluate each split, and write outputs.
results = []
for setting in scenarios:
    cohort = matched_cohort(setting["matching_ratio"], setting["control_replacement"])
    cohort["target_12m_source"] = cohort["target"]
    horizon_end = cohort["prediction_date"] + pd.DateOffset(months=setting["horizon_months"])
    observed_filing = cohort["bankruptcy_date"].notna()
    complete_negative_window = cohort["metadata_outcome_end_date"].ge(horizon_end)
    cohort = cohort.loc[observed_filing | complete_negative_window].copy()
    horizon_end = cohort["prediction_date"] + pd.DateOffset(months=setting["horizon_months"])
    cohort["target"] = (cohort["bankruptcy_date"].notna()
                        & cohort["bankruptcy_date"].le(horizon_end)).astype(int)
    changed_labels = int(cohort["target"].ne(cohort["target_12m_source"]).sum())
    predictors = feature_columns(setting["feature_set"], setting["missing_indicators"])
    categorical = [c for c in CATEGORICAL if c in cohort.columns]
    train = cohort.loc[cohort["split"].eq("train")]
    if train["target"].nunique() < 2:
        raise ValueError(f"Training split has one class for {setting['scenario']}")
    validation = cohort.loc[cohort["split"].eq("validation")]
    candidates = []
    for C_value in C_GRID:
        candidate = make_model(predictors, C_value)
        candidate.fit(train[predictors + categorical], train["target"],
                      classifier__sample_weight=train["sampling_weight"].to_numpy(dtype=float))
        validation_probability = candidate.predict_proba(validation[predictors + categorical])[:, 1]
        validation_metrics = score(validation["target"], validation_probability)
        candidates.append({"C": C_value, "model": candidate, **validation_metrics})
    selected = sorted(candidates, key=lambda row: (
        -row["pr_auc_average_precision"],
        -(row["roc_auc"] if pd.notna(row["roc_auc"]) else -np.inf),
        row["brier_score"], row["C"]))[0]
    model = selected["model"]
    for split in SPLITS:
        frame = cohort.loc[cohort["split"].eq(split)]
        probabilities = model.predict_proba(frame[predictors + categorical])[:, 1]
        summary = {
            **setting,
            "split": split,
            "events": len(frame),
            "companies": frame["company_id"].nunique(),
            "bankrupt_events": int(frame["target"].sum()),
            "prevalence": float(frame["target"].mean()) if len(frame) else np.nan,
            "mean_controls_per_case": float(frame["controls_per_case"].iloc[0]) if len(frame) else np.nan,
            "predictor_count": len(predictors),
            "labels_changed_from_12m": changed_labels,
            "selected_C": selected["C"],
            "validation_average_precision_for_selection": selected["pr_auc_average_precision"],
        }
        summary.update(score(frame["target"], probabilities))
        results.append(summary)

robustness_results = pd.DataFrame(results)
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
results_path = REPORTS / "robustness_results.csv"
robustness_results.to_csv(results_path, index=False)
test_results = robustness_results.loc[robustness_results["split"].eq("test")].copy()
plot_data = test_results.sort_values(["comparison", "scenario"], kind="stable")
positions = np.arange(len(plot_data))
fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True, constrained_layout=True)
for ax, metric, title in zip(axes, ["pr_auc_average_precision", "roc_auc"],
                            ["Test average precision", "Test ROC-AUC"]):
    ax.barh(positions, plot_data[metric], color="#3572a5")
    ax.set_yticks(positions, plot_data["scenario"])
    ax.set_xlabel(metric.replace("_", " ").title())
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    ax.invert_yaxis()
fig.suptitle("One-factor robustness comparisons on the chronological test split")
figure_path = FIGURES / "model_performance_comparison.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.close(fig)
print(f"Saved {len(robustness_results)} split-level results and the comparison figure.")


Saved 54 split-level results and the comparison figure.
